# Indusense — Sprint 1

Ce notebook produira les cartes d'identité des fichiers CSV placés dans `datas/`.

In [2]:
import csv
from codecs import BOM_UTF8
from collections.abc import Iterator
from datetime import date, datetime, time
from io import StringIO
from pathlib import Path

import pandas as pd
from sqlglot import exp, parse
from sqlglot.errors import ParseError, TokenError

from source_sql import DIALECTE_SQL, classer_colonne_sql, extraire_tables_sql

DATA_DIR = Path("datas")


def afficher_tableau(tableau: pd.DataFrame) -> None:
    """Utilise le rendu texte du notebook et laisse l’éditeur choisir les couleurs."""
    print(tableau.fillna("NULL").to_string(index=False))

## Lire les fichiers CSV

On conserve les valeurs brutes sous forme de texte pour préserver les identifiants. Les champs vides sont considérés comme manquants. L'encodage est déduit en essayant UTF-8, puis Windows-1252 et ISO-8859-1.

In [3]:
def lire_csv(chemin: Path) -> tuple[pd.DataFrame, str, str]:
    contenu = chemin.read_bytes()
    encodages = ("utf-8-sig",) if contenu.startswith(BOM_UTF8) else (
        "utf-8", "cp1252", "iso-8859-1"
    )
    for encodage in encodages:
        try:
            texte = contenu.decode(encodage)
        except UnicodeDecodeError:
            continue
        break
    else:
        raise ValueError(f"{chemin}: le fichier ne peut pas être décodé.")

    try:
        separateur = csv.Sniffer().sniff(texte[:8192], delimiters=",;\t").delimiter
    except csv.Error as erreur:
        raise ValueError(f"{chemin}: séparateur CSV indéterminé.") from erreur

    try:
        donnees = pd.read_csv(
            StringIO(texte),
            sep=separateur,
            dtype="string",
            keep_default_na=False,
            na_values=[""],
        )
    except (pd.errors.ParserError, pd.errors.EmptyDataError) as erreur:
        raise ValueError(f"{chemin}: lecture du CSV impossible ({erreur}).") from erreur

    return donnees, encodage, separateur

## Inférer les types et calculer les statistiques

Les colonnes connues suivent les règles explicites du projet. Pour les autres, toutes les valeurs présentes doivent être compatibles avec le type inféré. Les dates et heures inférées suivent les formats ISO. Les horodatages avec fuseau sont comparés en UTC. Une colonne mixte reste en texte et la carte indique la famille compatible avec le plus de valeurs. Les champs vides sont exclus de l’inférence et des statistiques.

In [4]:
IDENTIFIANTS = {"incident_id", "machine_id", "operator_badge"}
NUMERIQUES = {
    "releves_incidents.csv": {"severity"},
    "telemetry.csv": {
        "temperature_c", "pressure_bar", "voltage_mean_v",
        "rotation_mean_rpm", "pieces_produced",
    },
}
FORMATS_TEMPORELS = {
    "releves_incidents.csv": {
        "date": ("DATE", "%Y-%m-%d"),
        "time": ("TIME", "%H:%M"),
    },
    "telemetry.csv": {"timestamp": ("DATETIME", "%Y-%m-%d %H:%M:%S")},
}
COLONNES_INDICATEURS = {
    "type_surchauffe", "type_baisse_pression", "type_vibration",
    "type_bruit_mecanique", "type_surconsommation", "type_blocage_mecanique",
    "type_alarme_capteur", "type_arret_urgence", "type_defaut_qualite",
}
INDICATEURS = {"0": False, "1": True}
DATE_ISO = r"[0-9]{4}-[0-9]{2}-[0-9]{2}"
HEURE_ISO = r"(?:[01][0-9]|2[0-3]):[0-5][0-9](?::[0-5][0-9](?:\.[0-9]{1,6})?)?"
DATETIME_ISO = rf"{DATE_ISO}[T ]{HEURE_ISO}"
FUSEAU_ISO = r"(?:Z|[+-](?:[01][0-9]|2[0-3]):?[0-5][0-9])"


def convertir_numeriques(valeurs: pd.Series) -> pd.Series:
    """Écarte aussi les valeurs infinies des statistiques."""
    nombres = pd.to_numeric(valeurs, errors="coerce")
    return nombres.mask(nombres.isin([float("inf"), float("-inf")]))


def candidats_temporels(valeurs: pd.Series) -> Iterator[tuple[str, pd.Series]]:
    """Ne propose que des formats ISO non ambigus, sans mélanger les horloges."""
    for type_colonne, motif, avec_fuseau in (
        ("DATE", DATE_ISO, False),
        ("TIME", HEURE_ISO, False),
        ("DATETIME", DATETIME_ISO, False),
        ("DATETIME (avec fuseau)", DATETIME_ISO + FUSEAU_ISO, True),
    ):
        correspondantes = valeurs[valeurs.str.fullmatch(motif)]
        if correspondantes.empty:
            continue
        if type_colonne == "TIME":
            converties = correspondantes.map(time.fromisoformat)
        else:
            converties = pd.to_datetime(
                correspondantes, format="ISO8601", errors="coerce", utc=avec_fuseau
            )
        yield type_colonne, converties


def inferer_type(
    fichier: str, nom: str, valeurs: pd.Series, format_temporel: tuple[str, str] | None
) -> tuple[str, pd.Series, str, str]:
    """Renvoie le type, les valeurs converties, la provenance et le diagnostic."""
    explicite = "Règle explicite"
    observe = "Valeurs observées"
    nom_normalise = nom.casefold()
    if nom_normalise in IDENTIFIANTS or nom_normalise.endswith(("_id", "_badge")):
        return "IDENTITY", valeurs, explicite, ""
    if fichier == "releves_incidents.csv" and nom in COLONNES_INDICATEURS:
        return "BOOLEAN", valeurs.map(INDICATEURS), explicite, ""
    if format_temporel is not None:
        type_colonne, format_date = format_temporel
        dates = pd.to_datetime(valeurs, format=format_date, errors="coerce")
        return type_colonne, dates, explicite, ""
    if nom in NUMERIQUES.get(fichier, ()):
        return "NUMERIC", convertir_numeriques(valeurs), explicite, ""
    if valeurs.empty:
        return "TEXT", valeurs, observe, "Aucune valeur non vide"

    if valeurs.str.fullmatch(r"[0-9]+").all() and valeurs.str.match(r"0[0-9]").any():
        return "TEXT", valeurs, observe, "Zéros initiaux conservés"

    candidats = [("NUMERIC", convertir_numeriques(valeurs)), *candidats_temporels(valeurs)]
    type_probable, converties = max(candidats, key=lambda candidat: candidat[1].count())
    compatibles = converties.count()
    if compatibles == len(valeurs):
        return type_probable.partition(" ")[0], converties, observe, ""
    if compatibles:
        diagnostic = (
            f"Valeurs mixtes : {compatibles}/{len(valeurs)} compatibles avec {type_probable}"
        )
        return "TEXT", valeurs, observe, diagnostic
    return "TEXT", valeurs, observe, ""


def calculer_statistiques(
    type_colonne: str, valeurs: pd.Series, *, format_date: str | None = None
) -> dict[str, int | float | str]:
    if valeurs.empty:
        return {}
    if type_colonne == "NUMERIC":
        return {
            "min": valeurs.min().item(),
            "max": valeurs.max().item(),
            "mean": valeurs.mean().item(),
            "median": valeurs.median().item(),
        }
    if type_colonne in {"DATE", "TIME", "DATETIME"}:
        bornes = {"min": valeurs.min(), "max": valeurs.max()}
        if format_date is not None:
            return {cle: valeur.strftime(format_date) for cle, valeur in bornes.items()}
        if type_colonne == "DATE":
            return {cle: valeur.date().isoformat() for cle, valeur in bornes.items()}
        return {cle: valeur.isoformat() for cle, valeur in bornes.items()}
    return {}

## Décrire les deux fichiers

Chaque description regroupe les métadonnées, le type des colonnes et les statistiques. Les horaires sont ceux de la machine qui exécute le notebook. La section suivante utilise ces descriptions pour générer les cartes Markdown.

In [5]:
def decrire_csv(chemin: Path) -> dict[str, object]:
    debut = datetime.now().astimezone()
    source = chemin.resolve()
    taille = source.stat().st_size
    donnees, encodage, separateur = lire_csv(source)
    colonnes = []

    for nom, serie in donnees.items():
        # Un champ composé seulement d'espaces compte aussi comme manquant.
        valeurs = serie[serie.notna() & serie.str.strip().ne("")]
        format_temporel = FORMATS_TEMPORELS.get(source.name, {}).get(nom)
        type_colonne, converties, provenance, diagnostic = inferer_type(
            source.name, nom, valeurs, format_temporel
        )
        invalides = int(converties.isna().sum())
        if invalides:
            raise ValueError(
                f"{source.name}, colonne {nom}: {invalides} valeur(s) "
                f"invalide(s) pour le type {type_colonne}."
            )
        colonnes.append({
            "nom": nom,
            "type": type_colonne,
            "provenance": provenance,
            "diagnostic": diagnostic,
            "valeurs_manquantes": len(serie) - len(valeurs),
            "statistiques": calculer_statistiques(
                type_colonne, converties, format_date=format_temporel[1] if format_temporel else None
            ),
        })

    lignes, nombre_colonnes = donnees.shape
    return {
        "source": str(source),
        "taille_octets": taille,
        "debut": debut,
        "fin": datetime.now().astimezone(),
        "encodage": encodage,
        "separateur": separateur,
        "lignes": lignes,
        "nombre_colonnes": nombre_colonnes,
        "colonnes": colonnes,
    }

In [6]:
FICHIERS_CSV = ("releves_incidents.csv", "telemetry.csv")
descriptions = {nom: decrire_csv(DATA_DIR / nom) for nom in FICHIERS_CSV}

afficher_tableau(pd.DataFrame([
    {
        "Fichier": nom,
        "Encodage": description["encodage"],
        "Séparateur": repr(description["separateur"]),
        "Lignes": description["lignes"],
        "Colonnes": description["nombre_colonnes"],
    }
    for nom, description in descriptions.items()
]))

              Fichier Encodage Séparateur  Lignes  Colonnes
releves_incidents.csv    utf-8        ','    1245        18
        telemetry.csv    utf-8        ','  135626         7


## Générer les cartes d'identité Markdown

Chaque carte prend le nom du fichier source, la date et l'heure de début (AAAAMMJJ_HHMMSS) de son analyse. Les rapports sont enregistrés dans le dossier daté sous `artifacts/ingestion/identities/`.

In [7]:
IDENTITIES_DIR = Path("artifacts/ingestion/identities")


def formater_taille(octets: int) -> str:
    taille = float(octets)
    for unite in ("octets", "Ko", "Mo", "Go", "To"):
        if taille < 1_000 or unite == "To":
            return f"{taille:.2f} {unite}"
        taille /= 1_000


def rendre_entete_markdown(description: dict[str, object]) -> list[str]:
    source = Path(description["source"])
    debut = description["debut"]
    fin = description["fin"]
    return [
        f"# Carte d'identité — {source.name}",
        "",
        f"- Source: `{source}`",
        f"- Filsize: {formater_taille(description['taille_octets'])}",
        f"- ProcessBeginAt: {debut:%H:%M:%S}",
        f"- ProcessEndAt: {fin:%H:%M:%S}",
        f"- Charset: {description['encodage']}",
        f"- Rows Number: {description['lignes']}",
        f"- Cols Number: {description['nombre_colonnes']}",
    ]


def rendre_tableaux_colonnes(colonnes: list[dict], *, afficher_type_sql: bool = False) -> list[str]:
    niveau = "###" if afficher_type_sql else "##"
    lignes = [
        "",
        f"{niveau} Column list",
        "",
    ]
    if afficher_type_sql:
        lignes.extend([
            "| Colname | SQL Type | Inferred Type | Raison du classement |",
            "| --- | --- | --- | --- |",
        ])
    else:
        lignes.extend([
            "| Colname | Inferred Type | Origine | Diagnostic |",
            "| --- | --- | --- | --- |",
        ])
    statistiques = []
    for colonne in colonnes:
        nom = colonne["nom"].replace("\\", "\\\\").replace("|", "\\|")
        type_colonne = colonne["type"]
        if afficher_type_sql:
            lignes.append(
                f"| {nom} | {colonne['type_sql']} | {type_colonne} | {colonne['raison']} |"
            )
        else:
            provenance = colonne["provenance"]
            diagnostic = colonne["diagnostic"] or "—"
            lignes.append(f"| {nom} | {type_colonne} | {provenance} | {diagnostic} |")
        if type_colonne in {"NUMERIC", "DATE", "TIME", "DATETIME"}:
            valeurs = colonne["statistiques"]
            cellules = [str(valeurs.get(cle, "—")) for cle in ("min", "max", "mean", "median")]
            statistiques.append("| " + " | ".join([nom, *cellules]) + " |")

    if statistiques:
        lignes.extend([
            "",
            f"{niveau} Statistiques",
            "",
            "| Colname | Min value | Max value | Mean value | Median value |",
            "| --- | --- | --- | --- | --- |",
            *statistiques,
        ])
    return lignes


def rendre_carte_markdown(description: dict[str, object]) -> str:
    lignes = rendre_entete_markdown(description) + rendre_tableaux_colonnes(description["colonnes"])
    return "\n".join(lignes) + "\n"


def ecrire_carte_markdown(
    description: dict[str, object], destination: Path, contenu: str
) -> Path:
    """Enregistre une carte sans écraser un rapport existant."""
    debut = description["debut"]
    dossier = destination / f"{debut:%Y%m%d}"
    nom_source = Path(description["source"]).stem
    chemin = dossier / f"{nom_source}_{debut:%Y%m%d_%H%M%S}.md"
    dossier.mkdir(parents=True, exist_ok=True)
    with chemin.open("x", encoding="utf-8", newline="\n") as fichier:
        fichier.write(contenu)
    return chemin

In [8]:
cartes_markdown = {
    nom: ecrire_carte_markdown(description, IDENTITIES_DIR, rendre_carte_markdown(description))
    for nom, description in descriptions.items()
}

for chemin in cartes_markdown.values():
    print(chemin.as_posix())

artifacts/ingestion/identities/20261126/releves_incidents_20261126_154105.md
artifacts/ingestion/identities/20261126/telemetry_20261126_154105.md


## Décrire le script PostgreSQL

SQLGlot lit les définitions de tables et les lignes explicites des `INSERT … VALUES`. La carte indique si le classement vient d'une clé primaire, d'une référence, de la convention de nommage des identifiants ou du type déclaré. Les statistiques décrivent les valeurs présentes dans le script. Les valeurs par défaut calculées par PostgreSQL et les effets des clauses `ON CONFLICT` dépendent de l'exécution en base.

In [9]:
def profil_colonne_sql(colonne: exp.ColumnDef, lignes: list[dict]) -> dict:
    nom = colonne.name
    type_colonne, raison = classer_colonne_sql(colonne)
    valeurs = []
    for ligne in lignes:
        valeur = ligne.get(nom)
        if valeur is None or isinstance(valeur, exp.Null):
            continue
        if isinstance(valeur, exp.Literal):
            valeurs.append(valeur.this)
        elif isinstance(valeur, exp.Neg) and isinstance(valeur.this, exp.Literal):
            valeurs.append(f"-{valeur.this.this}")
        elif type_colonne in {"NUMERIC", "DATE", "TIME", "DATETIME"}:
            raise ValueError(f"Valeur SQL non littérale dans la colonne {nom}.")

    statistiques = {}
    if valeurs and type_colonne == "NUMERIC":
        nombres = convertir_numeriques(pd.Series(valeurs, dtype="string"))
        if nombres.isna().any():
            raise ValueError(f"Valeur numérique SQL invalide dans la colonne {nom}.")
        statistiques = calculer_statistiques(type_colonne, nombres)
    elif valeurs and type_colonne in {"DATE", "TIME", "DATETIME"}:
        if type_colonne == "DATE":
            dates = [date.fromisoformat(valeur) for valeur in valeurs]
        elif type_colonne == "TIME":
            dates = [time.fromisoformat(valeur) for valeur in valeurs]
        else:
            dates = [datetime.fromisoformat(valeur) for valeur in valeurs]
        statistiques = {"min": min(dates).isoformat(), "max": max(dates).isoformat()}

    return {
        "nom": nom,
        "type": type_colonne,
        "type_sql": colonne.kind.sql(dialect=DIALECTE_SQL),
        "raison": raison,
        "statistiques": statistiques,
    }


def decrire_sql(chemin: Path) -> dict:
    debut = datetime.now().astimezone()
    source = chemin.resolve()
    contenu = source.read_bytes()
    encodage = "utf-8-sig" if contenu.startswith(BOM_UTF8) else "utf-8"
    try:
        instructions = [
            instruction for instruction in parse(contenu.decode(encodage), read=DIALECTE_SQL)
            if instruction is not None
        ]
        tables = extraire_tables_sql(instructions)
        profils = [
            {
                "nom": nom,
                "lignes": len(table["lignes"]),
                "colonnes": [
                    profil_colonne_sql(colonne, table["lignes"])
                    for colonne in table["definitions"].values()
                ],
            }
            for nom, table in tables.items()
        ]
    except (ValueError, TokenError, ParseError) as erreur:
        raise ValueError(f"{source.name}: {erreur}") from erreur
    return {
        "source": str(source),
        "taille_octets": len(contenu),
        "debut": debut,
        "fin": datetime.now().astimezone(),
        "encodage": encodage,
        "instructions": len(instructions),
        "tables": profils,
        "lignes": sum(table["lignes"] for table in profils),
        "nombre_colonnes": sum(len(table["colonnes"]) for table in profils),
    }

In [10]:
def rendre_carte_sql(description: dict) -> str:
    lignes = rendre_entete_markdown(description)
    lignes.extend([
        "- Dialecte: PostgreSQL",
        f"- Instructions: {description['instructions']}",
        f"- Tables: {len(description['tables'])}",
        "",
        "Les lignes comptées correspondent aux tuples des INSERT. Les statistiques "
        "portent sur les valeurs explicites. Le symbole — indique une statistique "
        "indisponible ou sans objet, notamment pour les champs alimentés par DEFAULT NOW().",
    ])
    for table in description["tables"]:
        lignes.extend([
            "",
            f"## Table `{table['nom']}`",
            "",
            f"- Rows Number: {table['lignes']}",
            f"- Cols Number: {len(table['colonnes'])}",
            *rendre_tableaux_colonnes(table["colonnes"], afficher_type_sql=True),
        ])
    return "\n".join(lignes) + "\n"


description_sql = decrire_sql(DATA_DIR / "machine.sql.txt")
carte_sql = ecrire_carte_markdown(
    description_sql, IDENTITIES_DIR, rendre_carte_sql(description_sql)
)
print(carte_sql.as_posix())

artifacts/ingestion/identities/20261126/machine.sql_20261126_154106.md


# Étape 2 — Charger les sources dans Bronze

Les deux CSV et le script PostgreSQL alimentent quatre tables, car `machine.sql.txt` contient les données de `machine` et de `maintenance`. Le chargeur Python prépare les lignes, SQLGlot lit les `INSERT` sans exécuter le script, puis `psql` utilise `COPY` dans PostgreSQL. Chaque chargement remplace son jeu de données dans une transaction, ce qui permet de rejouer l'opération sans ajouter de lignes.

| Choix | Pourquoi |
| --- | --- |
| Valeurs sources en texte, suivies de leur colonne `_type` | Préserver la représentation d'origine, dont les zéros initiaux, tout en gardant le type métier visible. |
| Une colonne `hash` par ligne | Vérifier qu'un rejeu avec la même source et la même clé retrouve les mêmes empreintes. Le HMAC-SHA-256 porte sur les valeurs sources avant pseudonymisation. |
| Nom et badge pseudonymisés séparément par SHA-256 salé | Retirer les identifiants d'opérateur en clair de ces deux colonnes et préserver les associations observées. Le sel et la clé restent hors du dépôt. |
| Valeurs SQL non fournies conservées à `NULL` | Les champs remplis par un `DEFAULT` ne reçoivent pas de valeur inventée à l'ingestion. Une chaîne vide reste distincte de `NULL`. |

## 1. Démarrer PostgreSQL et préparer les secrets

Les cellules suivantes exécutent le chargement Bronze dans l’ordre. Docker Desktop doit être lancé et le fichier `docker-compose.yml` se trouve dans le dossier parent. Au premier lancement, le notebook crée le fichier local `.env.bronze`. Les exécutions suivantes réutilisent ses secrets, sans les afficher.

`CHARGER_BRONZE = False` laisse les tables en place et exécute seulement l’aperçu final. Pour recharger explicitement, mets `CHARGER_BRONZE = True` et relance les cellules Bronze depuis le début.

In [17]:
from inspect import getsource
from itertools import chain
from secrets import token_hex
from subprocess import CalledProcessError, run
from tempfile import TemporaryFile
from uuid import uuid4

from bronze import (
    charger_psql,
    commande_psql,
    encoder_champs,
    lire_csv_bronze,
    lire_secrets,
    lire_tables_bronze,
    lire_tables_machine,
    preparer_tables,
    sha256_source,
    transformer_ligne,
)


CHARGER_BRONZE = False
COMPOSE_BRONZE = Path("../docker-compose.yml")
SECRETS_BRONZE = Path(".env.bronze")
trace_chargement = {}


def preparer_secrets_bronze(chemin: Path) -> None:
    try:
        with chemin.open("x", encoding="ascii") as fichier:
            fichier.write(f"BRONZE_OPERATOR_SALT={token_hex(32)}\n")
            fichier.write(f"BRONZE_ROW_HASH_KEY={token_hex(32)}\n")
    except FileExistsError:
        # Réutiliser les mêmes valeurs assure la stabilité des empreintes.
        pass


if CHARGER_BRONZE:
    run(
        ["docker", "compose", "-f", str(COMPOSE_BRONZE), "up", "-d", "--wait",
         "--wait-timeout", "60", "db"],
        check=True,
    )
    preparer_secrets_bronze(SECRETS_BRONZE)
    variables_bronze = dict(
        ligne.split("=", 1)
        for ligne in SECRETS_BRONZE.read_text(encoding="utf-8-sig").splitlines()
        if "=" in ligne and not ligne.lstrip().startswith("#")
    )
    cle_bronze, sel_bronze = lire_secrets("releves_incidents", variables_bronze)
    print("PostgreSQL est prêt. Les secrets restent dans .env.bronze.")

PostgreSQL est prêt. Les secrets restent dans .env.bronze.


## 2. Lire les sources

Les CSV sont lus en texte ligne par ligne. SQLGlot associe les valeurs littérales des `INSERT` aux colonnes du `CREATE TABLE`, dans l’ordre du schéma. Un champ absent ou un `NULL` SQL est représenté par `None`. La première ligne lue est conservée dans le flux qui sera chargé.

In [18]:
sources_bronze = {
    "releves_incidents": DATA_DIR / "releves_incidents.csv",
    "telemetry": DATA_DIR / "telemetry.csv",
    "machine": DATA_DIR / "machine.sql.txt",
}

print(getsource(lire_csv_bronze) + "\n" + getsource(lire_tables_machine))

def empreintes_sources_bronze() -> dict[Path, str]:
    return {source: sha256_source(source) for source in sources_bronze.values()}


if CHARGER_BRONZE:
    empreintes_sources = empreintes_sources_bronze()
    source_chargee = {}
    tables_sources = []
    for dataset, source in sources_bronze.items():
        for nom, types, lignes in lire_tables_bronze(source, dataset):
            source_chargee[nom] = source
            lignes = iter(lignes)
            premiere = next(lignes, None)
            if premiere is None:
                raise ValueError(f"{source.name} : la table {nom} est vide.")
            tables_sources.append((nom, types, premiere, chain([premiere], lignes)))
    afficher_tableau(pd.DataFrame([
        {"Table": nom, "Colonnes sources": len(types), "Colonnes dans l’ordre": ", ".join(types)}
        for nom, types, _, _ in tables_sources
    ]))

def lire_csv_bronze(source: Path, types: dict[str, str]) -> Iterator[list[str]]:
    """Conserve les champs CSV sous forme de texte dans leur ordre d'origine."""
    with source.open(encoding="utf-8-sig", newline="") as fichier:
        lecteur = csv.reader(fichier, strict=True)
        if next(lecteur, None) != list(types):
            raise ValueError(f"{source.name} : en-tête différent du contrat Bronze.")
        for ligne in lecteur:
            if len(ligne) != len(types) or any("\x00" in valeur for valeur in ligne):
                raise ValueError(f"{source.name} : ligne CSV invalide à la ligne {lecteur.line_num}.")
            yield ligne

def lire_tables_machine(source: Path) -> Iterator[tuple[str, dict[str, str], list[list[Valeur]]]]:
    """Lit les littéraux des INSERT dans l’ordre du schéma, sans exécuter le dump."""
    try:
        instructions = [i for i in parse(source.read_text(encoding="utf-8-sig"), read=DIALECTE_SQL) if i]
        tables = extraire_tables_sql(instru

## 3. Fabriquer les valeurs Bronze

Le code affiché ci-dessous est celui utilisé pour chaque ligne. Il calcule le HMAC sur les valeurs d’origine encodées en JSON, remplace les champs d’opérateur par leurs SHA‑256 salés et ajoute après chaque valeur son type. Le tableau montre une ligne par table. La valeur source du badge est masquée et les empreintes sont raccourcies à l’affichage.

In [19]:
print(getsource(encoder_champs) + "\n" + getsource(transformer_ligne))

champs_demo = {
    "releves_incidents": "operator_badge",
    "telemetry": "temperature_c",
    "machine": "created_at",
    "maintenance": "duration_hours",
}


def enrichir_table_bronze(lignes, types, sel_operateur):
    for ligne in lignes:
        yield transformer_ligne(ligne, types, cle_bronze, sel_operateur=sel_operateur)


if CHARGER_BRONZE:
    tables_preparees = []
    exemples_prepares = []
    for nom, types, premiere_source, lignes in tables_sources:
        sel_operateur = sel_bronze if nom == "releves_incidents" else None
        preparees = enrichir_table_bronze(lignes, types, sel_operateur)
        premiere_bronze = next(preparees)
        champ = champs_demo[nom]
        index = list(types).index(champ)
        valeur_source = "[masquée]" if sel_operateur else premiere_source[index]
        valeur_bronze = premiere_bronze[2 * index]
        exemples_prepares.append({
            "Table": nom,
            "Champ": champ,
            "Source": valeur_source,
            "Bronze": valeur_bronze[:12] + "…" if sel_operateur and valeur_bronze else valeur_bronze,
            "_type": premiere_bronze[2 * index + 1],
            "hash": premiere_bronze[-1][:12] + "…",
        })
        tables_preparees.append((nom, types, chain([premiere_bronze], preparees)))
    afficher_tableau(pd.DataFrame(exemples_prepares))

def encoder_champs(champs: Sequence[Valeur]) -> bytes:
    """Encode une séquence ordonnée sans ambiguïté liée aux séparateurs."""
    return json.dumps(champs, ensure_ascii=False, separators=(",", ":")).encode("utf-8")

def transformer_ligne(
    ligne: Sequence[Valeur], types: dict[str, str], cle_hash: bytes,
    *, sel_operateur: bytes | None = None,
) -> list[Valeur]:
    """Ajoute les types et l’empreinte et pseudonymise si la source le demande."""
    empreinte = hmac.digest(cle_hash, encoder_champs(ligne), "sha256").hex()
    resultat = []
    for (colonne, type_colonne), valeur in zip(types.items(), ligne, strict=True):
        if sel_operateur is not None and colonne in CHAMPS_OPERATEUR and valeur:
            # Sépare les domaines nom et badge, même si une valeur se répète.
            valeur = hashlib.sha256(sel_operateur + b"\0" + encoder_champs([colonne, valeur])).hexdigest()
        resultat.extend((valeur, type_colonne))
    return [*resultat, empreinte]

            Tab

## 4. Écrire les lignes dans PostgreSQL

Les lignes enrichies sont préparées au format `COPY`. Une seule transaction remplace les quatre tables Bronze et ajoute leurs quatre entrées à l’historique. Chaque ligne reçoit ensuite le run_id du lot, sans l’inclure dans le hash. Si la préparation échoue, la commande SQL n’est pas envoyée. Si un chargement échoue en base, la transaction est annulée.

In [20]:
print(getsource(preparer_tables))

if CHARGER_BRONZE:
    lot_bronze = uuid4()
    with TemporaryFile() as flux:
        volumes_charges = preparer_tables(
            flux, tables_preparees, run_id=lot_bronze,
            sources=source_chargee, empreintes=empreintes_sources,
        )
        charger_psql(COMPOSE_BRONZE, flux)
    print(f"Lot enregistré dans bronze.ingestion_runs : {lot_bronze}")
    trace_chargement = {
        table: {"lignes": total, "sha256": empreintes_sources[source_chargee[table]]}
        for table, total in volumes_charges.items()
    }
    afficher_tableau(pd.DataFrame([
        {"Table": nom, "Lignes chargées": total} for nom, total in volumes_charges.items()
    ]))
else:
    print("Chargement ignoré. L’aperçu ci-dessous consulte les tables existantes.")

def preparer_tables(
    flux: BinaryIO, tables: Iterable[TableBronze], *, run_id: UUID,
    sources: Mapping[str, Path], empreintes: Mapping[Path, str],
) -> dict[str, int]:
    """Prépare le chargement des tables et l’historique dans la même transaction."""
    flux.write(b'BEGIN;\nCREATE SCHEMA IF NOT EXISTS "bronze";\n')
    volumes = {
        nom: ecrire_table_copy(
            flux, nom, types, lignes, run_id=run_id, autoriser_null=nom in TABLES_MACHINE
        )
        for nom, types, lignes in tables
    }
    for source in dict.fromkeys(sources.values()):
        if sha256_source(source) != empreintes[source]:
            raise ValueError(f"{source.name} a changé pendant la préparation du chargement.")
    ecrire_historique(flux, run_id, volumes, sources, empreintes)
    flux.write(b"COMMIT;\n")
    flux.seek(0)
    return volumes



Lot enregistré dans bronze.ingestion_runs : 98941e5d-36b2-4b32-8245-7489c8e2ee87
            Table  Lignes chargées
releves_incidents             1245
        telemetry           135626
          machine               15
      maintenance              115


## 5. Vérifier ce qui se trouve dans PostgreSQL

Cette dernière cellule compte les lignes et les colonnes directement en base. Elle affiche aussi un exemple de valeur stockée, son type, son empreinte et son run_id pour chaque table.

In [21]:
TABLES_BRONZE = ("releves_incidents", "telemetry", "machine", "maintenance")


def lire_bronze(requete: str) -> pd.DataFrame:
    """Lit le résultat d'une requête dans le PostgreSQL local."""
    sql = f"BEGIN READ ONLY;\nCOPY ({requete}) TO STDOUT WITH (FORMAT CSV, HEADER);\nCOMMIT;\n"
    resultat = run(
        commande_psql(COMPOSE_BRONZE, "-q"),
        input=sql.encode("utf-8"),
        capture_output=True,
        check=True,
    )
    return pd.read_csv(StringIO(resultat.stdout.decode("utf-8")), keep_default_na=False)


comptages = "\nUNION ALL\n".join(
    f"SELECT '{nom}' AS nom_table, count(*) AS lignes FROM bronze.{nom}"
    for nom in TABLES_BRONZE
)
requete_volumes = f"""
WITH volumes AS ({comptages})
SELECT nom_table AS "Table", lignes AS "Lignes", count(c.column_name) AS "Colonnes"
FROM volumes
JOIN information_schema.columns AS c
    ON c.table_schema = 'bronze' AND c.table_name = nom_table
GROUP BY nom_table, lignes
ORDER BY nom_table
"""

requete_exemples = """
(SELECT 'releves_incidents' AS "Table", incident_id AS "Identifiant",
        'operator_badge' AS "Champ", left(operator_badge, 12) || '…' AS "Valeur",
        operator_badge_type AS "Type", left(hash, 12) || '…' AS "Hash", run_id AS "Lot"
 FROM bronze.releves_incidents ORDER BY incident_id LIMIT 1)
UNION ALL
(SELECT 'telemetry', machine_id, 'temperature_c', temperature_c, temperature_c_type,
        left(hash, 12) || '…', run_id
 FROM bronze.telemetry WHERE temperature_c <> ''
 ORDER BY "timestamp", machine_id, hash LIMIT 1)
UNION ALL
(SELECT 'machine', machine_code, 'max_daily_capacity', max_daily_capacity,
        max_daily_capacity_type, left(hash, 12) || '…', run_id
 FROM bronze.machine ORDER BY machine_code LIMIT 1)
UNION ALL
(SELECT 'maintenance', maintenance_id, 'duration_hours', duration_hours,
        duration_hours_type, left(hash, 12) || '…', run_id
 FROM bronze.maintenance ORDER BY length(maintenance_id), maintenance_id LIMIT 1)
"""

try:
    volumes_bronze = lire_bronze(requete_volumes)
    exemples_bronze = lire_bronze(requete_exemples)
except (OSError, CalledProcessError) as erreur:
    print("Aperçu Bronze indisponible. Exécute la cellule de chargement ci-dessus.")
    detail = getattr(erreur, "stderr", None)
    if detail:
        print(detail.decode("utf-8", errors="replace").strip())
else:
    afficher_tableau(volumes_bronze)
    print()
    afficher_tableau(exemples_bronze)

            Table  Lignes  Colonnes
          machine      15        24
      maintenance     115        24
releves_incidents    1245        38
        telemetry  135626        16

            Table Identifiant              Champ        Valeur     Type          Hash                                  Lot
releves_incidents  INC-000001     operator_badge df3e1bdfdf46… IDENTITY 5dffdb6103ce… 98941e5d-36b2-4b32-8245-7489c8e2ee87
        telemetry     MACH-01      temperature_c        46.332  NUMERIC 32204f1643b0… 98941e5d-36b2-4b32-8245-7489c8e2ee87
          machine     MACH-01 max_daily_capacity           770  NUMERIC 1e8c54e29b6f… 98941e5d-36b2-4b32-8245-7489c8e2ee87
      maintenance           1     duration_hours          2.24  NUMERIC b61abd81b203… 98941e5d-36b2-4b32-8245-7489c8e2ee87


### Les cinq premières lignes de chaque table Bronze

Les incidents et les machines sont triés par identifiant, la télémétrie par horodatage puis machine, et la maintenance par identifiant numérique. Le hash départage les lignes ayant la même clé de tri.

Chaque extrait présente les **cinq lignes complètes**. Les champs et les colonnes `_type` sont affichés verticalement pour rester lisibles. Le `hash` et le `run_id` figurent en entier dans le tableau **Traçabilité**, avec les mêmes numéros de ligne. Pour les incidents, les deux champs opérateur sont affichés séparément sous leur forme pseudonymisée. `NULL` indique une valeur absente et `""` une chaîne vide.

In [4]:
from json import loads


ordre_extraits_bronze = {
    "releves_incidents": "incident_id, hash",
    "telemetry": '"timestamp", machine_id, hash',
    "machine": "machine_code, hash",
    "maintenance": "maintenance_id::bigint, hash",
}


def lire_cinq_lignes_bronze(table: str) -> pd.DataFrame:
    ordre = ordre_extraits_bronze[table]
    resultat = lire_bronze(f"""
        SELECT row_to_json(ligne)::text AS contenu
        FROM (SELECT * FROM bronze.{table} ORDER BY {ordre} LIMIT 5) AS ligne
        ORDER BY {ordre}
    """)
    # JSON distingue NULL de la chaîne vide et garde les champs textuels intacts.
    return pd.DataFrame([loads(contenu) for contenu in resultat["contenu"]], dtype=object)


def afficher_cinq_lignes_bronze(table: str, lignes: pd.DataFrame) -> None:
    print(f"\nbronze.{table}")
    if lignes.empty:
        print("Aucune ligne en base.")
        return

    pseudonymes = [col for col in ("operator_name", "operator_badge") if col in lignes]
    tracabilite = ["hash", "run_id"]
    donnees = lignes.drop(columns=[*pseudonymes, *tracabilite]).replace("", '""')
    colonnes = donnees.T.reset_index(names="Champ")
    colonnes.columns = ["Champ", *(f"Ligne {i}" for i in range(1, len(lignes) + 1))]
    afficher_tableau(colonnes)

    for champ in pseudonymes:
        print(f"\n{champ} (pseudonymisé)")
        afficher_tableau(pd.DataFrame({
            "Ligne": range(1, len(lignes) + 1),
            "Pseudonyme": lignes[champ].replace("", '""'),
        }))

    print("\nTraçabilité")
    metadata = lignes[tracabilite].copy()
    metadata.insert(0, "Ligne", range(1, len(lignes) + 1))
    afficher_tableau(metadata)


try:
    cinq_lignes_bronze = {
        table: lire_cinq_lignes_bronze(table) for table in ordre_extraits_bronze
    }
except (OSError, CalledProcessError):
    print("Extraits indisponibles. Démarre PostgreSQL et charge les quatre tables Bronze.")
else:
    for table, lignes in cinq_lignes_bronze.items():
        afficher_cinq_lignes_bronze(table, lignes)


bronze.releves_incidents
                      Champ          Ligne 1                       Ligne 2                 Ligne 3                   Ligne 4                  Ligne 5
                incident_id       INC-000001                    INC-000002              INC-000003                INC-000004               INC-000005
           incident_id_type         IDENTITY                      IDENTITY                IDENTITY                  IDENTITY                 IDENTITY
                       date       2025-06-01                    2025-06-01              2025-06-02                2025-06-03               2025-06-04
                  date_type             DATE                          DATE                    DATE                      DATE                     DATE
                       time            05:42                         21:08                   05:43                     05:43                    01:01
                  time_type             TIME                          TIME

## 6. Rapport de vérification Bronze

Le rapport associe chaque table au fichier traité et compare le nombre de lignes lues à celui observé en base. Le SHA‑256 identifie le fichier utilisé lors de cette exécution. Les empreintes répétées comptent les lignes en plus de la première occurrence de chaque hash. Bronze les conserve.

Les deux tableaux suivants recensent les champs vides ou `NULL` et vérifient les références machine et incident. Ces contrôles signalent les écarts sans modifier les données. Si le chargement est désactivé, le rapport montre l’état en base et laisse la preuve du dernier chargement non vérifiée.

In [22]:
source_par_table = {
    "releves_incidents": sources_bronze["releves_incidents"],
    "telemetry": sources_bronze["telemetry"],
    "machine": sources_bronze["machine"],
    "maintenance": sources_bronze["machine"],
}

requete_empreintes = "\nUNION ALL\n".join(f"""
SELECT '{table}' AS "Table", count(*) AS "Lignes en base",
       count(*) - count(DISTINCT hash) AS "Hash répétés",
       count(*) FILTER (WHERE hash !~ '^[0-9a-f]{{64}}$' OR hash IS NULL) AS "Hash invalides"
FROM bronze.{table}
""" for table in TABLES_BRONZE) + '\nORDER BY "Table"'


def requete_champs_vides(schema: pd.DataFrame) -> str:
    """Compte les champs vides en un seul parcours par table."""
    blocs = []
    for table, groupe in schema.groupby("table_name"):
        colonnes = groupe["column_name"].tolist()
        compteurs = ", ".join(
            f"""count(*) FILTER (WHERE "{col}" IS NULL OR "{col}" = '') AS "{col}" """
            for col in colonnes
        )
        valeurs = ", ".join(f"""('{col}', compte."{col}")""" for col in colonnes)
        blocs.append(f"""
            SELECT '{table}' AS "Table", colonne AS "Champ", absences AS "Vides ou NULL"
            FROM (SELECT {compteurs} FROM bronze.{table}) AS compte
            CROSS JOIN LATERAL (VALUES {valeurs}) AS detail(colonne, absences)
            WHERE absences > 0
        """)
    return "\nUNION ALL\n".join(blocs) + '\nORDER BY "Table", "Champ"'


liaisons_bronze = (
    ("telemetry", "machine_id", "machine", "machine_code"),
    ("releves_incidents", "machine_id", "machine", "machine_code"),
    ("maintenance", "machine_code", "machine", "machine_code"),
    ("maintenance", "related_incident_id", "releves_incidents", "incident_id"),
)
requete_references = "\nUNION ALL\n".join(f"""
SELECT '{table}.{champ}' AS "Référence", '{cible}.{cle}' AS "Cible",
       count(NULLIF(src.{champ}, '')) AS "Renseignées",
       count(*) FILTER (
           WHERE NULLIF(src.{champ}, '') IS NOT NULL
             AND NOT EXISTS (SELECT 1 FROM bronze.{cible} AS ref WHERE ref.{cle} = src.{champ})
       ) AS "Sans correspondance"
FROM bronze.{table} AS src
""" for table, champ, cible, cle in liaisons_bronze) + '\nORDER BY "Référence"'

requete_schema_bronze = """
SELECT source.table_name, source.column_name
FROM information_schema.columns AS source
JOIN information_schema.columns AS type_col
  ON type_col.table_schema = source.table_schema AND type_col.table_name = source.table_name
 AND type_col.ordinal_position = source.ordinal_position + 1
 AND type_col.column_name = source.column_name || '_type'
WHERE source.table_schema = 'bronze'
  AND source.table_name IN ('releves_incidents', 'telemetry', 'machine', 'maintenance')
  AND source.column_name ~ '^[a-z_][a-z0-9_]{0,62}$'
ORDER BY source.table_name, source.ordinal_position
"""

try:
    rapport_bronze = lire_bronze(requete_empreintes)
    schema_bronze = lire_bronze(requete_schema_bronze)
    champs_vides_bronze = lire_bronze(requete_champs_vides(schema_bronze))
    references_bronze = lire_bronze(requete_references)
except (OSError, CalledProcessError):
    print("Rapport indisponible. Démarre PostgreSQL et charge les quatre tables Bronze.")
else:
    traces = trace_chargement if CHARGER_BRONZE else {}
    rapport_bronze["Source"] = rapport_bronze["Table"].map(
        lambda table: source_par_table[table].name
    )
    rapport_bronze["SHA-256 source"] = rapport_bronze["Table"].map(
        lambda table: traces.get(table, {}).get("sha256", "—")
    )
    rapport_bronze["Lignes lues et chargées"] = rapport_bronze["Table"].map(
        lambda table: traces.get(table, {}).get("lignes", "—")
    )
    rapport_bronze["Vérification du volume"] = [
        "Non vérifié" if table not in traces else "OK" if attendu == observe else "Écart"
        for table, attendu, observe in zip(
            rapport_bronze["Table"], rapport_bronze["Lignes lues et chargées"],
            rapport_bronze["Lignes en base"], strict=True,
        )
    ]

    print("Volumes et contrôle des hash")
    afficher_tableau(rapport_bronze[[
        "Table", "Lignes lues et chargées", "Lignes en base", "Hash répétés",
        "Hash invalides", "Vérification du volume",
    ]])
    print("\nEmpreinte du fichier source")
    afficher_tableau(rapport_bronze[["Table", "Source", "SHA-256 source"]])
    print("\nChamps vides ou NULL")
    if champs_vides_bronze.empty:
        print("Aucun champ vide ou NULL.")
    else:
        afficher_tableau(champs_vides_bronze)
    print("\nRéférences entre tables")
    afficher_tableau(references_bronze)

Volumes et contrôle des hash
            Table  Lignes lues et chargées  Lignes en base  Hash répétés  Hash invalides Vérification du volume
          machine                       15              15             0               0                     OK
      maintenance                      115             115             0               0                     OK
releves_incidents                     1245            1245             0               0                     OK
        telemetry                   135626          135626             0               0                     OK

Empreinte du fichier source
            Table                Source                                                   SHA-256 source
          machine       machine.sql.txt 9eb877ae806515e852df46dd9f97f31aabfa8420a8f07dc50a5be4cefced8fcb
      maintenance       machine.sql.txt 9eb877ae806515e852df46dd9f97f31aabfa8420a8f07dc50a5be4cefced8fcb
releves_incidents releves_incidents.csv be7204b3320a39fa2c58f4be10b

## 7. Historique des chargements

PostgreSQL conserve dans `bronze.ingestion_runs` une entrée par table et par lot réussi. Les entrées d’un même lot partagent un identifiant. Chaque ligne de données le porte aussi dans run_id. Le lien vers l’historique se fait par run_id et nom de table. Chacune contient le SHA‑256 du fichier, le nombre de lignes et le commit Git courant s’il est disponible. Les modifications locales éventuelles ne sont pas identifiées par le commit.

Un rejeu ajoute un lot. Si le fichier source est identique, son SHA‑256 reste le même. `machine` et `maintenance` partagent le SHA‑256 du script. Dans les tables Bronze, deux lignes sources identiques restent présentes, même si elles ont le même hash de ligne.

In [23]:
requete_historique = """
SELECT run_id AS "Lot", loaded_at AT TIME ZONE 'UTC' AS "Chargé à (UTC)",
       target_table AS "Table", row_count AS "Lignes", source_file AS "Source",
       source_sha256 AS "SHA-256 source", git_commit AS "Commit Git"
FROM bronze.ingestion_runs
ORDER BY loaded_at DESC, run_id DESC, target_table
LIMIT 16
"""

lots_en_base = "\nUNION ALL\n".join(
    f"SELECT '{table}' AS nom, run_id, count(*) AS lignes FROM bronze.{table} GROUP BY run_id"
    for table in TABLES_BRONZE
)
requete_lots_actifs = f"""
WITH lots_actifs AS ({lots_en_base})
SELECT a.nom AS "Table", a.run_id AS "Lot des lignes", a.lignes AS "Lignes en base",
       h.row_count AS "Lignes du lot",
       CASE WHEN h.row_count = a.lignes THEN 'OK'
            WHEN h.run_id IS NULL THEN 'Lot manquant' ELSE 'Écart' END AS "Lien historique"
FROM lots_actifs AS a
LEFT JOIN bronze.ingestion_runs AS h ON h.run_id = a.run_id AND h.target_table = a.nom
ORDER BY a.nom
"""

try:
    historique_bronze = lire_bronze(requete_historique)
    lots_actifs_bronze = lire_bronze(requete_lots_actifs)
except (OSError, CalledProcessError):
    print("Historique indisponible. Exécute un premier chargement Bronze.")
else:
    print("Lots utilisés par les lignes en base")
    afficher_tableau(lots_actifs_bronze)
    print("\nHistorique des derniers chargements")
    afficher_tableau(historique_bronze)

Lots utilisés par les lignes en base
            Table                       Lot des lignes  Lignes en base  Lignes du lot Lien historique
          machine 98941e5d-36b2-4b32-8245-7489c8e2ee87              15             15              OK
      maintenance 98941e5d-36b2-4b32-8245-7489c8e2ee87             115            115              OK
releves_incidents 98941e5d-36b2-4b32-8245-7489c8e2ee87            1245           1245              OK
        telemetry 98941e5d-36b2-4b32-8245-7489c8e2ee87          135626         135626              OK

Historique des derniers chargements
                                 Lot             Chargé à (UTC)             Table  Lignes                Source                                                   SHA-256 source                               Commit Git
98941e5d-36b2-4b32-8245-7489c8e2ee87 2026-11-27 08:07:32.389072           machine      15       machine.sql.txt 9eb877ae806515e852df46dd9f97f31aabfa8420a8f07dc50a5be4cefced8fcb 18dd0cdf025411fac5b76

# Étape 3 — Diagnostic Silver en lecture seule

Cette étape prépare la discussion avec le métier. Elle observe toutes les lignes du Bronze courant dans une même transaction PostgreSQL en lecture seule. Elle ne crée aucune table Silver et ne retire, corrige ou impute aucune ligne. Le rechargement Bronze est désactivé par défaut dans la cellule de chargement ci-dessus.

Le [plan et registre des décisions](docs/silver/plan-et-decisions.md) précise les règles à faire valider. Les noms, badges et champs de commentaires ne sont pas lus pour ce diagnostic. Les exemples utilisent la table, le lot et le hash Bronze pour retrouver le contexte.

In [18]:
from collections import Counter, defaultdict
from decimal import Decimal, InvalidOperation
from json import loads
from re import fullmatch

from bronze import TYPES_INCIDENTS, TYPES_TELEMETRIE


# Formats sondés pour repérer des incompatibilités. Ils ne constituent pas des règles Silver approuvées.
champs_sondes = {
    "releves_incidents": {
        nom: type_colonne for nom, type_colonne in TYPES_INCIDENTS.items()
        if nom in {"incident_id", "machine_id"} or type_colonne not in {"IDENTITY", "TEXT"}
    },
    "telemetry": TYPES_TELEMETRIE,
    "machine": {
        "machine_code": "IDENTITY", "commissioning_date": "DATE",
        "max_daily_capacity": "NUMERIC", "max_hourly_capacity_pieces": "NUMERIC",
        "is_active": "BOOLEAN_SQL", "created_at": "DATETIME_OFFSET", "updated_at": "DATETIME_OFFSET",
    },
    "maintenance": {
        "maintenance_id": "IDENTITY", "machine_code": "IDENTITY",
        "related_incident_id": "IDENTITY", "maintenance_at": "DATETIME_OFFSET",
        "duration_hours": "NUMERIC", "created_at": "DATETIME_OFFSET", "updated_at": "DATETIME_OFFSET",
    },
}


def lire_snapshot_silver() -> tuple[dict, list, dict]:
    """Prend un instantané cohérent du Bronze, sans charger les champs personnels."""
    requetes = ["BEGIN ISOLATION LEVEL REPEATABLE READ READ ONLY;",
                "SELECT json_build_array('controle', current_setting('transaction_read_only'));"]
    lots_actifs = []

    for table, colonnes in champs_sondes.items():
        champs = ", ".join(f't."{nom}"' for nom in colonnes)
        requetes.append(
            f"SELECT json_build_array('ligne', '{table}', t.run_id, t.hash, {champs}) "
            f"FROM bronze.{table} AS t;"
        )
        lots_actifs.append(f"SELECT DISTINCT '{table}' AS nom, run_id FROM bronze.{table}")
        # Comparer le contenu uniquement dans les groupes au hash répété.
        requetes.append(f"""
            WITH repetes AS (
                SELECT run_id, hash FROM bronze.{table} GROUP BY run_id, hash HAVING count(*) > 1
            )
            SELECT json_build_array('hash_repete', '{table}', t.run_id, t.hash,
                                    count(DISTINCT (to_jsonb(t) - 'run_id' - 'hash')))
            FROM bronze.{table} AS t JOIN repetes AS r USING (run_id, hash)
            GROUP BY t.run_id, t.hash;
        """)

    actif = " UNION ALL ".join(lots_actifs)
    requetes.append(f"""
        SELECT json_build_array('manifeste', a.nom, a.run_id, h.source_file,
                                h.source_sha256, h.row_count)
        FROM ({actif}) AS a
        LEFT JOIN bronze.ingestion_runs AS h ON h.run_id = a.run_id AND h.target_table = a.nom
        ORDER BY a.nom, a.run_id;
    """)
    requetes.append("COMMIT;")
    resultat = run(
        commande_psql(COMPOSE_BRONZE, "-qAt"),
        input="\n".join(requetes).encode("utf-8"), capture_output=True, check=True,
    )

    lignes_par_table = {table: [] for table in champs_sondes}
    manifeste = []
    empreintes_repetees = {}
    lecture_seule = False
    for envoi in resultat.stdout.decode("utf-8").split("\n"):
        if not envoi.strip():
            continue
        genre, *donnees = loads(envoi)
        if genre == "controle":
            lecture_seule = donnees == ["on"]
        elif genre == "ligne":
            table, lot, empreinte, *valeurs = donnees
            colonnes = ["run_id", "hash", *champs_sondes[table]]
            lignes_par_table[table].append(dict(zip(colonnes, [lot, empreinte, *valeurs], strict=True)))
        elif genre == "manifeste":
            manifeste.append(donnees)
        elif genre == "hash_repete":
            table, lot, empreinte, contenus = donnees
            empreintes_repetees[(table, lot, empreinte)] = contenus

    if not lecture_seule:
        raise RuntimeError("PostgreSQL n’a pas confirmé le mode lecture seule.")
    return lignes_par_table, manifeste, empreintes_repetees


lignes_silver, manifeste_silver, hash_repetes_silver = lire_snapshot_silver()
volumes_par_lot = {
    (table, lot): total
    for table, lignes in lignes_silver.items()
    for lot, total in Counter(ligne["run_id"] for ligne in lignes).items()
}
rapport_volumes_silver = pd.DataFrame([
    {
        "Table": table, "Lot Bronze": lot, "Source": source, "SHA-256 source": sha,
        "Lignes lues": volumes_par_lot[(table, lot)], "Lignes du lot": espere,
    }
    for table, lot, source, sha, espere in manifeste_silver
])
print("Transaction PostgreSQL en lecture seule confirmée.")
afficher_tableau(rapport_volumes_silver)
print(f"Total diagnostiqué : {sum(map(len, lignes_silver.values()))} lignes · 0 ligne modifiée ou écartée.")

Transaction PostgreSQL en lecture seule confirmée.
            Table                           Lot Bronze                Source                                                   SHA-256 source  Lignes lues  Lignes du lot
          machine 98941e5d-36b2-4b32-8245-7489c8e2ee87       machine.sql.txt 9eb877ae806515e852df46dd9f97f31aabfa8420a8f07dc50a5be4cefced8fcb           15             15
      maintenance 98941e5d-36b2-4b32-8245-7489c8e2ee87       machine.sql.txt 9eb877ae806515e852df46dd9f97f31aabfa8420a8f07dc50a5be4cefced8fcb          115            115
releves_incidents 98941e5d-36b2-4b32-8245-7489c8e2ee87 releves_incidents.csv be7204b3320a39fa2c58f4be10bc5013fd74ab63aa4a5a5399fb0d151846110a         1245           1245
        telemetry 98941e5d-36b2-4b32-8245-7489c8e2ee87         telemetry.csv 68ec8dd09b691e6810d26626d17191414482552112e31cac04f110f3bb58fb6c       135626         135626
Total diagnostiqué : 137001 lignes · 0 ligne modifiée ou écartée.


## Motifs examinés

Un constat signale un champ ou un groupe à revoir. Une même ligne peut être concernée par plusieurs motifs. Les formats testés sont ceux déjà observés ou déclarés dans le Bronze : nombres finis, dates ISO, horodatages CSV sans fuseau, horodatages SQL avec décalage et indicateurs booléens. Les chaînes vides et `NULL` sont relevés tels quels.

Un couple machine/horodatage répété n’établit pas l’identité d’un relevé. Un dépassement de capacité est un signal à discuter, car la période couverte par le compteur reste à confirmer. Les valeurs physiquement impossibles ne sont pas évaluées tant qu’aucune limite n’a été validée.

In [19]:
motifs_silver = {
    "COPIE_EXACTE_CANDIDATE": "Hash répété et même contenu Bronze dans le lot",
    "HASH_REPETE_CONTENU_DIFFERENT": "Hash répété avec des contenus Bronze différents",
    "CONFLIT_IDENTIFIANT": "Même identifiant métier présumé et hash différents",
    "CLE_CANDIDATE_REPETEE": "Même machine et même horodatage dans la télémétrie",
    "CLE_ABSENTE_OU_INVALIDE": "Champ d’identification absent ou hors format sondé",
    "REFERENCE_INCONNUE": "Référence non vide absente du référentiel",
    "CONVERSION_IMPOSSIBLE": "Valeur non vide hors du format sondé",
    "CHAMP_VIDE_OU_NULL": "Champ sondé vide ou NULL",
    "CAPACITE_DEPASSEE": "Nombre de pièces supérieur à la capacité déclarée",
    "VALEUR_IMPOSSIBLE": "Limite physique ou métier à faire valider",
}


def est_absente(valeur: str | None) -> bool:
    return valeur is None or valeur == ""


def valeur_lisible(valeur: str, type_colonne: str) -> bool:
    """Essaie le format de diagnostic sans remplacer la valeur source."""
    try:
        match type_colonne:
            case "NUMERIC":
                return bool(fullmatch(r"[+-]?(?:[0-9]+(?:\.[0-9]*)?|\.[0-9]+)(?:[eE][+-]?[0-9]+)?", valeur)) and Decimal(valeur).is_finite()
            case "DATE":
                return bool(fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}", valeur)) and bool(date.fromisoformat(valeur))
            case "TIME":
                return bool(fullmatch(r"[0-9]{2}:[0-9]{2}", valeur)) and bool(time.fromisoformat(valeur))
            case "DATETIME" | "DATETIME_OFFSET":
                format_iso = r"[0-9]{4}-[0-9]{2}-[0-9]{2}[ T][0-9]{2}:[0-9]{2}:[0-9]{2}(?:\.[0-9]+)?"
                if type_colonne == "DATETIME_OFFSET":
                    format_iso += r"(?:Z|[+-][0-9]{2}(?::?[0-9]{2})?)"
                return bool(fullmatch(format_iso, valeur)) and bool(datetime.fromisoformat(valeur))
            case "BOOLEAN":
                return valeur in {"0", "1"}
            case "BOOLEAN_SQL":
                return valeur.casefold() in {"true", "false"}
            case "IDENTITY":
                return bool(valeur.strip())
            case _:
                raise ValueError(f"Sonde non définie pour {type_colonne}")
    except (ValueError, InvalidOperation, OverflowError):
        return False


def analyser_motifs_silver(lignes_par_table: dict, empreintes_repetees: dict) -> tuple:
    compteurs = Counter()
    lignes_touchees = defaultdict(set)
    exemples = defaultdict(list)
    groupes = Counter()
    cles = {
        "releves_incidents": ("incident_id",),
        "telemetry": ("machine_id", "timestamp"),
        "machine": ("machine_code",),
        "maintenance": ("maintenance_id",),
    }
    machines = {l["machine_code"] for l in lignes_par_table["machine"] if not est_absente(l["machine_code"])}
    incidents = {l["incident_id"] for l in lignes_par_table["releves_incidents"] if not est_absente(l["incident_id"])}
    references = {
        "releves_incidents": (("machine_id", machines),),
        "telemetry": (("machine_id", machines),),
        "maintenance": (("machine_code", machines), ("related_incident_id", incidents)),
    }
    capacites = defaultdict(set)
    for machine in lignes_par_table["machine"]:
        valeur = machine["max_hourly_capacity_pieces"]
        if not est_absente(valeur) and valeur_lisible(valeur, "NUMERIC"):
            capacites[machine["machine_code"]].add(Decimal(valeur))

    def signaler(motif, table, position, ligne, champ, valeur, constat):
        compteurs[motif, table, champ] += 1
        lignes_touchees[motif].add((table, position))
        if len(exemples[motif]) < 3:
            exemples[motif].append({
                "Table": table, "Champ": champ, "Valeur": valeur,
                "Constat": constat, "Lot": ligne["run_id"], "Hash Bronze": ligne["hash"],
            })

    for table, lignes in lignes_par_table.items():
        par_cle = defaultdict(list)
        for position, ligne in enumerate(lignes):
            champs_invalides = set()
            for champ, type_colonne in champs_sondes[table].items():
                valeur = ligne[champ]
                if est_absente(valeur):
                    signaler("CHAMP_VIDE_OU_NULL", table, position, ligne, champ,
                             "NULL" if valeur is None else '""', "Valeur source absente")
                    champs_invalides.add(champ)
                elif not valeur_lisible(valeur, type_colonne):
                    signaler("CONVERSION_IMPOSSIBLE", table, position, ligne, champ,
                             valeur, f"Format sondé : {type_colonne}")
                    champs_invalides.add(champ)

            for champ in cles[table]:
                if champ in champs_invalides:
                    signaler("CLE_ABSENTE_OU_INVALIDE", table, position, ligne, champ,
                             repr(ligne[champ]), "Champ de la clé observée")
            if not champs_invalides.intersection(cles[table]):
                cle = tuple(ligne[champ] for champ in cles[table])
                par_cle[ligne["run_id"], cle].append((position, ligne))

            for champ, valeurs_connues in references.get(table, ()):
                if not est_absente(ligne[champ]) and ligne[champ] not in valeurs_connues:
                    signaler("REFERENCE_INCONNUE", table, position, ligne, champ,
                             ligne[champ], "Aucune cible trouvée dans le Bronze courant")

            variants = empreintes_repetees.get((table, ligne["run_id"], ligne["hash"]))
            if variants is not None:
                motif = "COPIE_EXACTE_CANDIDATE" if variants == 1 else "HASH_REPETE_CONTENU_DIFFERENT"
                signaler(motif, table, position, ligne, "hash", ligne["hash"], f"{variants} contenu(s) distinct(s)")

            if table == "telemetry" and "pieces_produced" not in champs_invalides:
                seuils = capacites.get(ligne["machine_id"], set())
                if len(seuils) == 1 and Decimal(ligne["pieces_produced"]) > next(iter(seuils)):
                    signaler("CAPACITE_DEPASSEE", table, position, ligne, "pieces_produced",
                             ligne["pieces_produced"], f"{ligne['machine_id']} : capacité déclarée {next(iter(seuils))}")

        for (_, cle), membres in par_cle.items():
            if len(membres) < 2:
                continue
            if table == "telemetry":
                motif = "CLE_CANDIDATE_REPETEE"
            elif len({ligne["hash"] for _, ligne in membres}) > 1:
                motif = "CONFLIT_IDENTIFIANT"
            else:
                continue
            groupes[motif] += 1
            for position, ligne in membres:
                signaler(motif, table, position, ligne, " + ".join(cles[table]),
                         " / ".join(cle), f"{len(membres)} lignes dans le groupe")

    return compteurs, lignes_touchees, exemples, groupes


comptes_silver, touches_silver, exemples_silver, groupes_silver = analyser_motifs_silver(
    lignes_silver, hash_repetes_silver
)

## À discuter avec le métier

Le bilan donne le nombre de lignes distinctes touchées par motif. Le détail compte les signalements par champ. Les trois exemples au maximum par motif montrent la valeur telle qu’elle est lue dans Bronze. Lorsqu’aucun cas n’est détecté, il n’y a aucun exemple à présenter.

Les points à trancher portent sur l’identité d’un relevé, le fuseau des CSV, la période couverte par le compteur de pièces, les codes d’absence et les seuils réellement applicables. Toute règle qui changerait les valeurs, leur interprétation ou les lignes retenues devra être validée et consignée avec son responsable, sa date et son périmètre.

In [20]:
bilan_silver = pd.DataFrame([
    {
        "Motif": motif,
        "Lignes touchées": "—" if motif == "VALEUR_IMPOSSIBLE" else len(touches_silver[motif]),
        "Groupes repérés": groupes_silver.get(motif, "—"),
        "Statut": "Non évalué" if motif == "VALEUR_IMPOSSIBLE" else
                 "À valider" if touches_silver[motif] else "Aucun cas détecté",
    }
    for motif in motifs_silver
])
afficher_tableau(bilan_silver)

print("\nDétail par table et champ")
afficher_tableau(pd.DataFrame([
    {"Motif": motif, "Table": table, "Champ": champ, "Signalements": total}
    for (motif, table, champ), total in sorted(comptes_silver.items())
], columns=["Motif", "Table", "Champ", "Signalements"]))

for motif, description in motifs_silver.items():
    print(f"\n{motif} — {description}")
    if exemples_silver[motif]:
        afficher_tableau(pd.DataFrame(exemples_silver[motif]))
    elif motif == "VALEUR_IMPOSSIBLE":
        print("Pas d’exemple : règle métier non validée.")
    else:
        print("Pas d’exemple sur ce lot.")

print("\nDécisions appliquées aux données : aucune.")

                        Motif Lignes touchées Groupes repérés            Statut
       COPIE_EXACTE_CANDIDATE               0               — Aucun cas détecté
HASH_REPETE_CONTENU_DIFFERENT               0               — Aucun cas détecté
          CONFLIT_IDENTIFIANT               0               — Aucun cas détecté
        CLE_CANDIDATE_REPETEE            2686            1340         À valider
      CLE_ABSENTE_OU_INVALIDE               0               — Aucun cas détecté
           REFERENCE_INCONNUE               0               — Aucun cas détecté
        CONVERSION_IMPOSSIBLE               0               — Aucun cas détecté
           CHAMP_VIDE_OU_NULL            2958               —         À valider
            CAPACITE_DEPASSEE           37136               —         À valider
            VALEUR_IMPOSSIBLE               —               —        Non évalué

Détail par table et champ
                Motif       Table                  Champ  Signalements
    CAPACITE_DEPASSEE 

# Étape 4 — Produire le Silver validé

Le [registre métier](docs/silver/plan-et-decisions.md) valide la première occurrence de chaque relevé dans l’ordre du CSV, la fenêtre horaire UTC et la conservation des relevés avec mesures à `NULL`, sans imputation. Les champs opérateur et le commentaire d’incident restent dans Bronze.

Le traitement relit Bronze en lecture seule, vérifie le lot et le SHA‑256 du CSV, puis rapproche les lignes par leur hash. Il conserve les valeurs de la première occurrence, journalise les réémissions et signale les périodes manquantes. Les dépassements de capacité et les candidats atypiques restent dans Silver. Les tables Silver sont remplacées ensemble dans une seule transaction. `CHARGER_BRONZE` reste désactivé.

In [21]:
from silver import (
    VERSION_CONTRAT,
    choisir_telemetrie,
    construire_silver,
    ecrire_artefacts,
    lire_bronze_silver,
    publier_silver,
)


CHARGER_SILVER = True
DOSSIER_SILVER = Path("artifacts/silver") / VERSION_CONTRAT

if CHARGER_SILVER:
    instantane = lire_bronze_silver(COMPOSE_BRONZE)
    config_silver = dict(
        ligne.split("=", 1)
        for ligne in SECRETS_BRONZE.read_text(encoding="utf-8-sig").splitlines()
        if "=" in ligne and not ligne.lstrip().startswith("#")
    )
    cle_silver, _ = lire_secrets("telemetry", config_silver)
    premieres_occurrences, journal_reemissions = choisir_telemetrie(
        instantane.lignes["telemetry"], DATA_DIR / "telemetry.csv",
        instantane.lots["telemetry"]["source_sha256"], cle_silver,
    )
    donnees_silver = construire_silver(instantane, premieres_occurrences, journal_reemissions)

    print("Bilan avant publication")
    afficher_tableau(pd.DataFrame(donnees_silver["manifeste"])[[
        "source_table", "bronze_lignes", "silver_lignes", "reemissions_ecartees",
    ]])
    print("\nExemples de décisions sur les réémissions")
    afficher_tableau(pd.DataFrame(journal_reemissions).head(3))

Bilan avant publication
     source_table  bronze_lignes  silver_lignes  reemissions_ecartees
        telemetry         135626         134280                  1346
releves_incidents           1245           1245                     0
          machine             15             15                     0
      maintenance            115            115                     0

Exemples de décisions sur les réémissions
                       bronze_run_id machine_id                 timestamp  ligne_conservee                                                    hash_conserve  ligne_ecartee                                                      hash_ecarte                          motif
98941e5d-36b2-4b32-8245-7489c8e2ee87    MACH-01 2025-06-01 00:00:00+00:00                1 45d2e441c1888836b9d63003910187609850a6e220accd12d55680fa8f2c6018              2 32204f1643b096f9165b19581d817f7a8d3369fc84da076f4e0c89f9c60a9000 REEMISSION_PREMIERE_OCCURRENCE
98941e5d-36b2-4b32-8245-7489c8e2ee87    MACH-01 2

## Publier et vérifier le Silver

Les tables de qualité conservent les motifs et les liens avec Bronze. Le journal `reemissions` contient le numéro d’enregistrement dans le CSV, sans compter l’en-tête, et les hash des deux occurrences. Les périodes manquantes sont des intervalles début inclus, fin exclue, regroupés par machine et mesure. Les heures sans relevé sont détectées entre le premier et le dernier relevé horaire observés par machine. L’état d’arrêt reste inconnu sans historique d’état fiable.

Le signal statistique de revue est calculé par machine et par mesure, à plus de trois écarts interquartiles sous le premier quartile ou au-dessus du troisième. Il ne définit aucune limite physique. Une valeur non convertible selon le contrat arrête la publication du lot et laisse les valeurs disponibles en Bronze pour décision métier.

In [22]:
if CHARGER_SILVER:
    publier_silver(COMPOSE_BRONZE, donnees_silver)
    ecrire_artefacts(DOSSIER_SILVER, donnees_silver)
    verification_silver = lire_bronze("""
        SELECT 'telemetry' AS "Table", count(*) AS "Lignes en base" FROM silver.telemetry
        UNION ALL SELECT 'releves_incidents', count(*) FROM silver.releves_incidents
        UNION ALL SELECT 'machine', count(*) FROM silver.machine
        UNION ALL SELECT 'maintenance', count(*) FROM silver.maintenance
        UNION ALL SELECT 'reemissions', count(*) FROM silver.reemissions
        UNION ALL SELECT 'periodes_manquantes', count(*) FROM silver.periodes_manquantes
        UNION ALL SELECT 'heures_sans_releve', count(*) FROM silver.heures_sans_releve
        ORDER BY "Table"
    """)
    afficher_tableau(verification_silver)
    assert all(
        total == len(donnees_silver[nom])
        for nom, total in zip(verification_silver["Table"], verification_silver["Lignes en base"], strict=True)
    )
    print("\nExemples de périodes manquantes (UTC)")
    afficher_tableau(pd.DataFrame(donnees_silver["periodes_manquantes"]).head(3))
    print("\nRapport qualité")
    print((DOSSIER_SILVER / "rapport.md").read_text(encoding="utf-8"))
else:
    print("Publication Silver désactivée. Le diagnostic ci-dessus reste consultable.")

              Table  Lignes en base
 heures_sans_releve               0
            machine              15
        maintenance             115
periodes_manquantes             364
        reemissions            1346
  releves_incidents            1245
          telemetry          134280

Exemples de périodes manquantes (UTC)
                       bronze_run_id machine_id       mesure                 debut_utc                   fin_utc  nombre_releves
98941e5d-36b2-4b32-8245-7489c8e2ee87    MACH-01 pressure_bar 2025-06-19 21:00:00+00:00 2025-06-20 08:00:00+00:00              11
98941e5d-36b2-4b32-8245-7489c8e2ee87    MACH-01 pressure_bar 2025-07-02 19:00:00+00:00 2025-07-03 00:00:00+00:00               5
98941e5d-36b2-4b32-8245-7489c8e2ee87    MACH-01 pressure_bar 2025-09-09 10:00:00+00:00 2025-09-09 20:00:00+00:00              10

Rapport qualité
# Rapport qualité Silver

Contrat : silver-1.0

| Table | Bronze | Silver | Réémissions écartées |
| --- | ---: | ---: | ---: |
| telemetry 